# GP2 Bioinformatics Training Workshop Argentina  

## Rare-Variant Burden Analysis Using WGS Data

<span style="background-color: #90EE90">GP2 ❤️ Open Science 😍</span>

**Module:** Ancestry-stratified rare-variant burden analysis using GP2 whole-genome sequencing data

**Author:** Konstantin Senkevich, on behalf of the Global Parkinson's Genetics Program (GP2; ASAP)

**Example gene:** LRRK2

**Methods:** SKAT-O, CMC-Wald and single-variant logistic regression

**Estimated Computation and Runtime:**

**Estimated Specifications:** Default; 8 CPU, 30 GB Memory, 100 GB Persistent Disk Size  
**Estimated Runtime:** Approximately 1–2 hrs.  
**Date Last Updated:** 22/07/2026

## Workflow

This notebook demonstrates an ancestry-stratified rare-variant burden analysis using GP2 WGS data.

**1. Set up the environment**

- Load Python packages
- Define GP2 Release 12 paths
- Install the required software
- Define the gene and ancestry groups

**2. Prepare phenotype and covariate files**

- Select WGS Parkinson’s disease cases and controls
- Encode phenotype and sex
- Add age and PC1–PC5
- Remove related individuals
- Create ancestry-specific covariate files
- Match phenotype IDs to PLINK and VCF sample IDs

**3. Extract and annotate the gene**

- Extract the gene region from each ancestry dataset
- Export the genotypes to VCF
- Annotate variants using ANNOVAR
- Define functional variant masks

**4. Run burden analyses**

- Extract variants included in each mask
- Run SKAT-O and CMC-Wald
- Merge ancestry-specific results
- Review nominally significant findings

**5. Review individual variants**

- Run single-variant logistic regression
- Compare allele frequencies in cases and controls
- Add functional annotation
- Remove intronic variants from the final summary

**Important:** Frequency thresholds, covariates, variant masks and statistical tests should be defined before reviewing association results.

# Covariates and Sample IDs

The burden and single-variant analyses are adjusted for:

- sex
- age at sample collection
- PC1–PC5

The principal components are included to account for population structure within each ancestry group.

## Binary phenotype coding

- `PHENO = 2`: Parkinson's disease case
- `PHENO = 1`: control
- `SEX = 1`: male
- `SEX = 2`: female

Participants with missing values in any model covariate are excluded from the corresponding association test.

Two covariate-file formats are used:

- **PLINK:** `FID` and `IID` match the original PGEN sample IDs
- **RVTests:** `IID` matches the sample names in the exported VCF

Sample IDs must match exactly between genotype, phenotype and covariate files.

## Loading Python libraries

In [ ]:
# Getting Started
## Import packages
# File and system operations
import os
import sys
import pathlib
import subprocess
import urllib.parse

# Data handling
import pandas as pd
import numpy as np
from io import StringIO

# Plotting
import matplotlib.pyplot as plt
%matplotlib inline

# Notebook display
from IPython.display import display, HTML

# Google BigQuery
from google.cloud import bigquery

In [ ]:
## GP2 v12.0

# Define the Release 12 root path and display the available files
# in the clinical_data and meta_data directories.
REL12_PATH = pathlib.Path(pathlib.Path.home(), 'workspace/gp2_tier2_eu_release12')

!ls -hal {REL12_PATH}/clinical_data
!ls -hal {REL12_PATH}/meta_data

In [ ]:
# Define Release paths
EXTENDED_CLINICAL_DATA_PATH = pathlib.Path(REL12_PATH, 'clinical_data/r12_extended_clinical_data_vwb.csv')
CLINICAL_DATA_PATH = pathlib.Path(REL12_PATH, 'clinical_data/master_key_release12_final_vwb.csv')
RELATED_DATA_PATH = pathlib.Path(REL12_PATH, 'meta_data/related_samples/')

# Raw Genotyping
RAW_GENO_PATH = pathlib.Path(REL12_PATH, 'raw_genotypes')

# Imputed
IMPUTED_GENO_PATH = pathlib.Path(REL12_PATH, 'imputed_genotypes')

# WGS
WGS_DATA_PATH = pathlib.Path(REL12_PATH, "wgs/dragen_joint_calling/plink")
PCS_GENO_PATH = pathlib.Path(REL12_PATH, "wgs/dragen_joint_calling/pcs")
WGS_RELATED_PATH = pathlib.Path(REL12_PATH, "wgs/dragen_joint_calling/related_samples")

In [ ]:
trainee_id = "your_name"

In [ ]:
## Define gene and hg38 coordinates

gene = "LRRK2"
chromosome = 12
gene_start = 40224997
gene_end = 40369285

WORK_DIR = pathlib.Path(
    f"/home/jupyter/workspace/ws_files/BOTH_LEVELS_WORKSHOP/BURDEN_ANALYSIS_{trainee_id}"
)

PROJECT_DIR = WORK_DIR / gene
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

ancestries = ['AFR', 'AAC', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'MDE', 'SAS', 'CAH']

for ancestry in ancestries:
    (PROJECT_DIR / ancestry).mkdir(parents=True, exist_ok=True)

print("Gene:", gene)
print("Region:", f"chr{chromosome}:{gene_start}-{gene_end}")
print("Project directory:", PROJECT_DIR)

## Install Packages

In [ ]:
%%capture
%%bash

# Install plink 1.9
cd /home/jupyter/
if test -e /home/jupyter/plink; then
    echo "Plink is already installed in /home/jupyter/"
else
    echo "Plink is not installed"
    cd /home/jupyter

    wget https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20250615.zip 

    unzip -o plink_linux_x86_64_20250615.zip
    mv plink plink1.9
fi

In [ ]:
%%bash

# chmod plink 1.9 to make sure you have permission to run the program
chmod u+x /home/jupyter/plink1.9

In [ ]:
%%capture
%%bash

# Install plink 2.0
cd /home/jupyter/
if test -e /home/jupyter/plink2; then

echo "Plink2 is already installed in /home/jupyter/"
else
echo "Plink2 is not installed"
cd /home/jupyter/

wget http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip

unzip -o plink2_linux_x86_64_latest.zip

fi

In [ ]:
%%bash

# chmod plink 2 to make sure you have permission to run the program
chmod u+x /home/jupyter/plink2

In [ ]:
%%bash
set -e

ANNOVAR_ROOT="/home/jupyter/workspace/ws_files/Reference/annovar"
ANNOVAR_DIR="${ANNOVAR_ROOT}/software"
ANNOVAR_URL="http://www.openbioinformatics.org/annovar/download/0wgxR2rIVP/annovar.latest.tar.gz"

mkdir -p "${ANNOVAR_DIR}"

if [ -f "${ANNOVAR_DIR}/table_annovar.pl" ]; then
    echo "ANNOVAR is already installed."
else
    echo "Installing ANNOVAR programs..."

    ARCHIVE=$(mktemp --suffix=.tar.gz /tmp/annovar.XXXXXX)

    wget -q "${ANNOVAR_URL}" -O "${ARCHIVE}"
    tar -xzf "${ARCHIVE}" \
        -C "${ANNOVAR_DIR}" \
        --strip-components=1

    rm -f "${ARCHIVE}"

    echo "ANNOVAR installation complete."
fi

In [ ]:
#ANNOVAR WAS PREINSTALLED FOR YOU ALONG WITH ALL OF THE RELEVANT PACKAGES; PLEASE SEE CODE TO INSTALL MOST OF THEM BELOW

#%%capture
#%%bash

# Install ANNOVAR: Download resources for annotation

#cd /home/jupyter/workspace/ws_files/Reference/annovar/software
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar refGene humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar clinvar_20170905 humandb/
#perl annotate_variation.pl -buildver hg38 -downdb cytoBand humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar ensGene humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar exac03 humandb/ 
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar avsnp147 humandb/ 
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar dbnsfp47a humandb/ #latest version of dbNSFP
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad211_genome humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar ljb26_all humandb/

In [ ]:
%%bash

cd "/home/jupyter/workspace/ws_files/Reference/annovar/humandb"
ls

In [ ]:
%%bash

# Install RVTESTS: Option 1 (~15min)
if [ ! -e /home/jupyter/tooles/rvtests ]; then
    echo "RVTESTS not found. Installing..."
    mkdir -p /home/jupyter/tools/rvtests
    cd /home/jupyter/tools/rvtests

    wget https://github.com/zhanxw/rvtests/releases/download/v2.1.0/rvtests_linux64.tar.gz 

    tar -zxvf rvtests_linux64.tar.gz
else
    echo "RVTESTS is already installed."
fi

In [ ]:
%%bash
cd /home/jupyter/tools/rvtests/executable
ls

In [ ]:
! chmod 777 /home/jupyter/tools/rvtests/executable/rvtest

In [ ]:
%%bash

/home/jupyter/tools/rvtests/executable/rvtest --help

## Create a covariate file with GP2 WGS data

### STEP 1. Load clinical data and define phenotype

In [ ]:
# STEP 1. Load clinical data and define phenotype

df = pd.read_csv(CLINICAL_DATA_PATH, low_memory=False)

# Use NBA ancestry when available; otherwise use the WGS ancestry label
df["label"] = df["nba_label"].fillna(df["wgs_label"])

# RVTests phenotype coding: 2 = PD, 1 = control
df["PHENO"] = np.select(
    [
        df["GP2_PHENO"].isin(["Affected_PD", "PD"]),
        # df["GP2_PHENO"].isin(["PD"]),
        df["GP2_PHENO"].isin(["Population Control", "Control"])
        # df["GP2_PHENO"].isin(["Control"])
    ],
    [2, 1],
    default=np.nan
)

df["PHENO"].value_counts(dropna=False)

### STEP 2. Keep WGS cases and controls

In [ ]:
# STEP 2. Keep WGS cases and controls

key = df.loc[
    (df["wgs"] == 1)
    & df["PHENO"].isin([1, 2]),
    [
        "GP2ID",
        "biological_sex_for_qc",
        "age_at_sample_collection",
        "age_of_onset",
        "label",
        "PHENO"
    ]
].copy()

key = key.rename(columns={
    "GP2ID": "IID",
    "biological_sex_for_qc": "SEX",
    "age_at_sample_collection": "AGE",
    "age_of_onset": "AAO"
})

key["SEX"] = key["SEX"].map({
    "Male": 1,
    "Female": 2
})

key["AGE"] = pd.to_numeric(key["AGE"], errors="coerce")
key["AAO"] = pd.to_numeric(key["AAO"], errors="coerce")
key["PHENO"] = key["PHENO"].astype(int)

print("PD cases:", (key["PHENO"] == 2).sum())
print("Controls:", (key["PHENO"] == 1).sum())
print("Total:", len(key))
print("Missing SEX:", key["SEX"].isna().sum())
print("Missing AGE:", key["AGE"].isna().sum())
print("Missing AAO:", key["AAO"].isna().sum())
print("Missing AAO among cases:", key.loc[key["PHENO"] == 2, "AAO"].isna().sum())

### STEP 3. Load ancestry-specific PCs

In [ ]:
# STEP 3. Load ancestry-specific PCs

ancestries = [
    "AFR", "AAC", "AJ", "AMR", "CAS",
    "EAS", "EUR", "MDE", "SAS", "CAH"
]

pc_tables = {}

for ancestry in ancestries:
    
    pc_path = pathlib.Path(
        PCS_GENO_PATH,
        ancestry,
        f"{ancestry}_release12.eigenvec"
    )

    if not pc_path.exists():
        print(f"Missing PC file for {ancestry}: {pc_path}")
        continue

    pcs = pd.read_csv(
        pc_path,
        sep="\t",
        header=None
    )

    # Keep sample ID and the first five PCs
    pcs = pcs.iloc[:, 1:7].copy()

    pcs.columns = [
        "IID",
        "PC1",
        "PC2",
        "PC3",
        "PC4",
        "PC5"
    ]

    pc_tables[ancestry] = pcs

    print(f"{ancestry}: {len(pcs)} samples with PCs")

### STEP 4. Remove second-degree-or-closer relatives within each ancestry

Kinship threshold should be 0.0884

In [ ]:
# STEP 4. Remove second-degree-or-closer relatives within each ancestry

import pathlib
import networkx as nx
import pandas as pd

ancestries = ["AFR", "AAC", "AJ", "AMR", "CAH", "CAS", "EAS", "EUR", "MDE", "SAS"]

KINSHIP_THRESHOLD = 0.0884
PHENOTYPE_COLUMN = "PHENO"
CASE_VALUE = 2

covariate_tables = {}
related_tables = {}
close_pair_tables = {}
remove_id_tables = {}
relatedness_summaries = {}

### STEP 4A. Merge covariates and read the relatedness files

In [ ]:
# STEP 4A. Merge covariates and read the relatedness files

for ancestry in ancestries:
    if ancestry not in pc_tables:
        print(f"{ancestry}: PC table not found")
        continue

    covariates = key[key["label"] == ancestry].merge(
        pc_tables[ancestry],
        on="IID",
        how="inner",
        validate="one_to_one"
    )

    related_path = pathlib.Path(
        WGS_RELATED_PATH,
        ancestry,
        f"{ancestry}_release12.related"
    )

    if not related_path.exists():
        print(f"{ancestry}: relatedness file not found")
        covariate_tables[ancestry] = covariates
        continue

    related = pd.read_csv(related_path, sep=",", dtype=str)
    related.columns = related.columns.str.lstrip("#").str.strip()
    related["KINSHIP"] = pd.to_numeric(related["KINSHIP"], errors="coerce")

    covariate_tables[ancestry] = covariates
    related_tables[ancestry] = related

    print(
        f"{ancestry}: {len(covariates)} samples; "
        f"{len(related)} relatedness records"
    )

### STEP 4B. Select second-degree-or-closer related pairs

In [ ]:
# STEP 4B. Select second-degree-or-closer related pairs

for ancestry, related in related_tables.items():
    covariates = covariate_tables[ancestry]

    close_pairs = related[
        related["KINSHIP"] > KINSHIP_THRESHOLD
    ].copy()

    for column in ["IID1", "IID2"]:
        close_pairs[column] = (
            close_pairs[column]
            .str.replace("_s1", "", regex=False)
            .str.strip()
        )

    close_pairs = close_pairs.dropna(subset=["IID1", "IID2"])
    close_pairs = close_pairs[close_pairs["IID1"] != close_pairs["IID2"]]

    analysis_ids = set(covariates["IID"])

    close_pairs = close_pairs[
        close_pairs["IID1"].isin(analysis_ids)
        & close_pairs["IID2"].isin(analysis_ids)
    ].copy()

    close_pair_tables[ancestry] = close_pairs

    print(
        f"{ancestry}: {len(close_pairs)} "
        "second-degree-or-closer pairs"
    )

### STEP 4C. Select a fast maximal unrelated subset

In [ ]:
# STEP 4C. Select a fast maximal unrelated subset

for ancestry, close_pairs in close_pair_tables.items():
    covariates = covariate_tables[ancestry]

    related_graph = nx.from_pandas_edgelist(
        close_pairs,
        source="IID1",
        target="IID2"
    )

    phenotype = (
        covariates
        .set_index("IID")[PHENOTYPE_COLUMN]
        .to_dict()
    )

    # Prioritize cases first.
    # Within cases or controls, prioritize samples with fewer relatives.
    ranked_ids = sorted(
        related_graph.nodes,
        key=lambda iid: (
            phenotype.get(iid) == CASE_VALUE,
            -related_graph.degree(iid)
        ),
        reverse=True
    )

    keep_related_ids = set()

    for iid in ranked_ids:
        # Retain the sample only if none of its relatives
        # has already been retained.
        if not any(
            relative in keep_related_ids
            for relative in related_graph.neighbors(iid)
        ):
            keep_related_ids.add(iid)

    remove_ids = set(related_graph.nodes) - keep_related_ids

    remove_id_tables[ancestry] = remove_ids

    print(
        f"{ancestry}: "
        f"{related_graph.number_of_nodes()} related samples, "
        f"{len(keep_related_ids)} retained, "
        f"{len(remove_ids)} removed"
    )

### STEP 4D. Remove selected samples and verify that no related pairs remain

In [ ]:
# STEP 4D. Remove selected samples and verify that no related pairs remain

for ancestry, covariates in covariate_tables.items():
    close_pairs = close_pair_tables.get(
        ancestry,
        pd.DataFrame(columns=["IID1", "IID2"])
    )
    remove_ids = remove_id_tables.get(ancestry, set())

    before = len(covariates)

    covariates = covariates[
        ~covariates["IID"].isin(remove_ids)
    ].copy()

    remaining_ids = set(covariates["IID"])

    remaining_pairs = close_pairs[
        close_pairs["IID1"].isin(remaining_ids)
        & close_pairs["IID2"].isin(remaining_ids)
    ]

    assert remaining_pairs.empty, (
        f"{ancestry}: related pairs remain after pruning"
    )

    covariate_tables[ancestry] = covariates

    print(
        f"{ancestry}: {before} before, "
        f"{before - len(covariates)} removed, "
        f"{len(covariates)} retained, "
        f"{len(remaining_pairs)} related pairs remaining"
    )

In [ ]:
for ancestry, covariates in covariate_tables.items():
    covariates["FID"] = 0
    covariates["FATID"] = 0
    covariates["MATID"] = 0

    final_df = covariates[[
        "FID", "IID", "FATID", "MATID", "SEX", "AGE", "AAO",
        "PHENO", "PC1", "PC2", "PC3", "PC4", "PC5"
    ]].copy()

    output_dir = PROJECT_DIR / ancestry
    output_dir.mkdir(parents=True, exist_ok=True)

    covariate_path = output_dir / f"{ancestry}_covariate.txt"
    keep_path = output_dir / f"{ancestry}.samplestokeep"

    final_df.to_csv(
        covariate_path,
        sep="\t",
        index=False,
        na_rep="NA"
    )

    final_df[["FID", "IID"]].rename(
        columns={"FID": "#FID"}
    ).to_csv(
        keep_path,
        sep="\t",
        index=False
    )

    print(f"{ancestry}: {len(final_df)} samples saved")

In [ ]:
gene = "LRRK2"
chromosome = 2
gene_start = 171783405
gene_end = 171894244


ancestries = ["AFR", "AJ", "AAC", "AMR", "CAS", "EAS", "EUR", "MDE", "SAS", "CAH"]
freqs = [0.01]
variant_classes = ["potentially_functional", "coding_variants", "loss_of_function", "CADD_gt20"]
test_types = ["SkatO"]
#test_types = ["SkatO", "CMCWald"]

# Store all gene-specific outputs inside the trainee working directory
PROJECT_DIR = pathlib.Path(WORK_DIR) / gene
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

for ancestry in ancestries:
    (PROJECT_DIR / ancestry).mkdir(parents=True, exist_ok=True)

print(f"Gene: {gene}")
print(f"Region: chr{chromosome}:{gene_start}-{gene_end} (hg38)")
print(f"Project directory: {PROJECT_DIR}")

In [ ]:
summary_records = []

for ancestry in ancestries:
    cov_path = PROJECT_DIR / ancestry / f"{ancestry}_covariate.txt"

    if not cov_path.exists():
        continue

    df_cov = pd.read_csv(cov_path, sep="\t", low_memory=False)

    for column in ["AGE", "AAO", "SEX", "PHENO"]:
        df_cov[column] = pd.to_numeric(df_cov[column], errors="coerce")

    summary_records.append({
        "Ancestry": ancestry,
        "Total": len(df_cov),
        "PD_cases": (df_cov["PHENO"] == 2).sum(),
        "Controls": (df_cov["PHENO"] == 1).sum(),
        "Mean_age": round(df_cov["AGE"].mean(), 1),
        "SD_age": round(df_cov["AGE"].std(), 1),
        "Missing_AGE": df_cov["AGE"].isna().sum(),
        "Missing_AAO": df_cov["AAO"].isna().sum(),
        "Male": (df_cov["SEX"] == 1).sum(),
        "Female": (df_cov["SEX"] == 2).sum(),
        "Missing_SEX": df_cov["SEX"].isna().sum()
    })

summary_df = pd.DataFrame(summary_records)
summary_df = summary_df.sort_values("Ancestry").reset_index(drop=True)
summary_df

## Extraction and preparation for annotation of the gene

### Define the gene and genomic region
* The example analysis uses `LRRK2` hg38: `chr12:40224997-40369285`
* The coordinates below use the hg38 genome build. Change the gene name and genomic coordinates when adapting the notebook to another gene.
* Make sure you are working in YOUR folder

In [ ]:
## Define gene and hg38 coordinates
gene = "LRRK2"
chromosome = 12
gene_start = 40224997
gene_end = 40369285

WORK_DIR = pathlib.Path(
    f"/home/jupyter/workspace/ws_files/BOTH_LEVELS_WORKSHOP/BURDEN_ANALYSIS_{trainee_id}")

PROJECT_DIR = WORK_DIR / gene
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

ancestries = ['AFR', 'AAC', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'MDE', 'SAS', 'CAH']

for ancestry in ancestries:
    (PROJECT_DIR / ancestry).mkdir(parents=True, exist_ok=True)

print("Gene:", gene)
print("Region:", f"chr{chromosome}:{gene_start}-{gene_end}")
print("Project directory:", PROJECT_DIR)

In [ ]:
## Extract gene region using PLINK

ancestries = ['AFR', 'AAC', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'MDE', 'SAS', 'CAH']

for ancestry in ancestries:
    ancestry_path = PROJECT_DIR / ancestry
    input_prefix = WGS_DATA_PATH / ancestry / f"chr{chromosome}_{ancestry}_release12"
    output_prefix = ancestry_path / f"{ancestry}_{gene}"

    print(f"\nExtracting {gene} in {ancestry}")

    # Extract the previously defined hg38 gene region
    ! /home/jupyter/plink2 \
    --pfile {input_prefix} \
    --chr {chromosome} \
    --from-bp {gene_start} \
    --to-bp {gene_end} \
    --make-bed \
    --out {output_prefix}

## Turn binary files into VCF

In [ ]:
## Export PLINK files to compressed VCF

genes = [gene]
ancestries = ['AFR', 'AAC', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'MDE', 'SAS', 'CAH']

for ancestry in ancestries:
    ancestry_path = PROJECT_DIR / ancestry
    input_prefix = ancestry_path / f"{ancestry}_{genes[0]}"
    output_prefix = ancestry_path / f"{ancestry}_{'_'.join(genes)}"

    print(f"\nProcessing {ancestry}")

    # Check that the extracted PLINK files are available
    if not input_prefix.with_suffix('.bed').exists():
        print(f"Missing: {input_prefix}.bed")
        continue

    # Export the gene region to a compressed VCF
    ! /home/jupyter/plink2 \
    --bfile {input_prefix} \
    --export vcf id-paste=iid vcf-dosage=HDS bgz \
    --out {output_prefix}

In [ ]:
## Index VCF files

for ancestry in ancestries:
    ancestry_path = PROJECT_DIR / ancestry
    vcf_path = ancestry_path / f"{ancestry}_{'_'.join(genes)}.vcf.gz"

    # Index the VCF for ANNOVAR and RVTests
    if not vcf_path.exists():
        print(f"Missing: {vcf_path}")
        continue

    !tabix -f -p vcf {vcf_path}
    print(f"Indexed: {vcf_path.name}")

## Annotate using ANNOVAR

### Annotate the extracted variants

A VCF contains variant positions and genotypes, but it does not necessarily
describe the biological effect of each variant. ANNOVAR adds information that
can be used to create functional variant masks.

In this step, we add:

- **refGene annotation** to identify the affected gene and consequence, such as
  missense, synonymous, stop-gain or splice-site.
- **ClinVar annotation** to identify variants previously submitted with clinical
  classifications.
- **dbNSFP annotation** to add computational prediction scores, including scores
  used to prioritize potentially damaging variants.

The annotation databases do not independently prove that a variant is
pathogenic. Their results should be interpreted together with allele frequency,
variant quality, segregation evidence and the relevant phenotype.

In [ ]:
%%bash

# Copy only the ANNOVAR programs to an executable local directory

mkdir -p /home/jupyter/annovar

cp /home/jupyter/workspace/ws_files/Reference/annovar/software/*.pl \
   /home/jupyter/annovar/

chmod 755 /home/jupyter/annovar/*.pl

echo "Local ANNOVAR programs are ready."

In [ ]:
## Annotate variants with ANNOVAR

# ANNOVAR adds functional information to each variant.
# Here, refGene provides gene-based consequences and dbNSFP provides
# computational pathogenicity prediction scores.

import os
import pathlib

# Run the ANNOVAR programs from the local executable directory
ANNOVAR_DIR = pathlib.Path("/home/jupyter/annovar")

# Read the large reference databases from the shared directory
ANNOVAR_DB = pathlib.Path(
    "/home/jupyter/workspace/ws_files/Reference/annovar/humandb"
)

# Stop early and provide a clear message if the setup is incomplete
assert os.access(
    ANNOVAR_DIR / "annotate_variation.pl",
    os.X_OK,
), "Run the local ANNOVAR setup cell before continuing."

assert (
    ANNOVAR_DIR / "table_annovar.pl"
).exists(), "table_annovar.pl was not found."

assert ANNOVAR_DB.exists(), "The shared ANNOVAR database directory was not found."

print(f"ANNOVAR programs: {ANNOVAR_DIR}")
print(f"ANNOVAR references: {ANNOVAR_DB}")


# Run annotation separately for each ancestry group
for ancestry in ancestries:

    # Each ancestry has its own results directory
    ancestry_path = PROJECT_DIR / ancestry

    # Compressed VCF created in the previous step
    vcf_path = ancestry_path / f"{ancestry}_{gene}.vcf.gz"

    # Intermediate file required by ANNOVAR
    avinput_path = ancestry_path / f"{ancestry}_{gene}.annovar.avinput"

    # Prefix for the ANNOVAR output files
    out_prefix = ancestry_path / f"{ancestry}_{gene}.annovar"

    print(f"\nAnnotating {ancestry}")

    # Skip the ancestry if its input VCF is missing
    if not vcf_path.exists():
        print(f"Missing VCF: {vcf_path}")
        continue

    # Decompress the VCF and convert it to ANNOVAR input format.
    # The original compressed VCF is not deleted.
    convert_cmd = (
        f'gzip -dc "{vcf_path}" | '
        f'perl "{ANNOVAR_DIR}/convert2annovar.pl" '
        f'-format vcf4old - > "{avinput_path}"'
    )

    # Add two annotations:
    #
    # refGene: gene name and functional consequence
    # dbNSFP: computational pathogenicity prediction scores
    #
    # Operation codes:
    # g = gene-based annotation
    # f = filter-based annotation
    annotate_cmd = (
        f'perl "{ANNOVAR_DIR}/table_annovar.pl" '
        f'"{avinput_path}" "{ANNOVAR_DB}" '
        f'-buildver hg38 '
        f'-out "{out_prefix}" '
        f'-remove '
        f'-protocol refGene,dbnsfp47a '
        f'-operation g,f '
        f'-nastring . '
        f'-polish'
    )

    # Convert the VCF
    if os.system(convert_cmd) != 0:
        print(f"Conversion failed for {ancestry}")
        continue

    # Annotate the variants
    if os.system(annotate_cmd) != 0:
        print(f"Annotation failed for {ancestry}")
        continue

    print(f"Annotation completed for {ancestry}")

In [ ]:
## Create annotation-based variant masks

genes = [gene]
gene_label = "_".join(genes)
cols = ['Chr', 'Start', 'End', 'Gene.refGene']

for ancestry in ancestries:
    ancestry_path = PROJECT_DIR / ancestry
    file_path = ancestry_path / f"{ancestry}_{gene_label}.annovar.hg38_multianno.txt"
    print(f"\nWorking on {ancestry}, {gene_label}")

    if not file_path.exists():
        print(f"Missing file: {file_path}")
        continue

    # Read annotations and keep variants assigned to the selected gene
    gene_df = pd.read_csv(file_path, sep='\t')
    gene_df = gene_df[gene_df['Gene.refGene'].str.contains("|".join(genes), na=False)].copy()
    gene_df['CADD_phred'] = pd.to_numeric(gene_df['CADD_phred'], errors='coerce')
    print(f"Total variants: {len(gene_df)}")

    # Define annotation masks
    potentially_functional = gene_df[gene_df['Func.refGene'] != 'intronic']
    coding_variants = gene_df[(gene_df['Func.refGene'] == 'splicing') | ((gene_df['Func.refGene'] == 'exonic') & (gene_df['ExonicFunc.refGene'] != 'synonymous SNV'))]
    loss_of_function = gene_df[(gene_df['Func.refGene'] == 'splicing') | gene_df['ExonicFunc.refGene'].isin(['stopgain', 'startloss', 'frameshift deletion', 'frameshift insertion'])]
    cadd_gt20 = gene_df[gene_df['CADD_phred'] > 20]

    # Remove duplicate transcript annotations and save PLINK range files
    prefix = ancestry_path / f"{ancestry}_{gene_label}"
    masks = {'potentially_functional': potentially_functional, 'coding_variants': coding_variants, 'loss_of_function': loss_of_function, 'CADD_gt20': cadd_gt20}

    for mask_name, mask_df in masks.items():
        variants = mask_df[cols].drop_duplicates()
        variants.to_csv(f"{prefix}.{mask_name}.variantstoKeep.txt", sep='\t', index=False, header=False)
        print(f"{mask_name}: {len(variants)} variants")

In [ ]:
## Extract VCFs for each annotation mask

for ancestry in ancestries:
    ancestry_path = PROJECT_DIR / ancestry
    vcf_input = ancestry_path / f"{ancestry}_{gene}.vcf.gz"

    if not vcf_input.exists():
        print(f"Missing VCF: {vcf_input}")
        continue

    for variant_class in variant_classes:
        extract_file = ancestry_path / f"{ancestry}_{gene}.{variant_class}.variantstoKeep.txt"
        output_prefix = ancestry_path / f"{ancestry}_{gene}.{variant_class}"

        print(f"\nExtracting {ancestry}, {variant_class}")

        # Skip masks without a variant list
        if not extract_file.exists() or extract_file.stat().st_size == 0:
            print(f"No variants: {extract_file}")
            continue

        # Extract the selected variants and export them as a compressed VCF
        cmd = (
            f"/home/jupyter/plink2 "
            f"--vcf {vcf_input} dosage=HDS "
            f"--extract range {extract_file} "
            f"--export vcf id-paste=iid vcf-dosage=HDS bgz "
            f"--out {output_prefix}"
        )

        if os.system(cmd) != 0:
            print(f"Extraction failed: {ancestry}, {variant_class}")
            continue

        # Index the output VCF for RVTests
        if os.system(f"tabix -f -p vcf {output_prefix}.vcf.gz") != 0:
            print(f"Indexing failed: {ancestry}, {variant_class}")

## Burden Analyses using RVTests

## Run gene-based burden tests
* RVTests requires a gene definition file in refFlat format. Upload `refFlat.txt` to `/home/jupyter/refFlat.txt` before running this step.
* For each ancestry and annotation mask, we run:
    - SKAT-O
    - CMC-Wald  

Both tests are adjusted for sex, age and PC1-PC5. The rare-variant threshold is applied by RVTests using `--freqUpper`.

In [ ]:
## Extract the refFlat gene-definition file
!gunzip -c "/home/jupyter/workspace/ws_files/Reference/refFlat.txt.gz" > "/home/jupyter/workspace/ws_files/Reference/refFlat.txt"

In [ ]:
## Run SKAT-O and CMC-Wald burden tests

import os

gene = 'LRRK2'
freqs = [0.01]
freq = [0.01]
variant_classes = ['potentially_functional', 'coding_variants', 'loss_of_function', 'CADD_gt20']
ancestries = ['AFR', 'AAC', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'MDE', 'SAS', 'CAH']

# Upload refFlat.txt to /home/jupyter/refFlat.txt before running this cell
refFlat_file = pathlib.Path('/home/jupyter/workspace/ws_files/Reference/refFlat.txt')
if not refFlat_file.exists():
    raise FileNotFoundError('Upload refFlat.txt to /home/jupyter/workspace/ws_files/refFlat.txt before running RVTests.')

for ancestry in ancestries:
    ancestry_path = PROJECT_DIR / ancestry
    pheno_file = ancestry_path / f"{ancestry}_covariate.txt"

    for freq in freqs:
        freq_label = f"{freq:.3f}".rstrip('0').rstrip('.')

        for variant_class in variant_classes:
            # Each VCF contains variants from one annotation mask
            vcf_input = ancestry_path / f"{ancestry}_{gene}.{variant_class}.vcf.gz"
            output_prefix = ancestry_path / f"{ancestry}.freq{freq_label}.burden.{gene}.{variant_class}"

            print(f"\nRunning {ancestry}, {variant_class}, MAF ≤ {freq}")

            if not vcf_input.exists():
                print(f"Missing VCF: {vcf_input}")
                continue

            # PHENO and covariates are stored in the same RVTests-formatted file
            if not pheno_file.exists():
                print(f"Missing covariate file: {pheno_file}")
                continue

            # Run SKAT-O and CMC-Wald, adjusted for sex, age and PC1-PC5
            cmd = (
                f"/home/jupyter/tools/rvtests/executable/rvtest "
                f"--noweb --hide-covar "
                f"--out {output_prefix} "
                f"--kernel skato "
                #f"--burden cmcWald "
                f"--freqUpper {freq} "
                f"--inVcf {vcf_input} "
                f"--pheno {pheno_file} "
                f"--pheno-name PHENO "
                f"--gene {gene} "
                f"--geneFile {refFlat_file} "
                f"--covar {pheno_file} "
                f"--covar-name SEX,AGE,PC1,PC2,PC3,PC4,PC5"
            )

            if os.system(cmd) != 0:
                print(f"RVTests failed: {ancestry}, {variant_class}")

In [ ]:
## Merge SKAT-O results across annotation masks

import os
import pandas as pd

gene = 'LRRK2'
mafs = [0.01]
variant_classes = ['potentially_functional', 'coding_variants', 'loss_of_function', 'CADD_gt20']
ancestries = ['AFR', 'AAC', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'MDE', 'SAS', 'CAH']

for ancestry in ancestries:
    ancestry_path = PROJECT_DIR / ancestry
    merged_results = []

    for maf in mafs:
        freq_label = f"{maf:.3f}".rstrip('0').rstrip('.')

        for variant_class in variant_classes:
            # Locate the SKAT-O result for this ancestry, MAF threshold and mask
            assoc_path = ancestry_path / f"{ancestry}.freq{freq_label}.burden.{gene}.{variant_class}.SkatO.assoc"

            if not assoc_path.exists():
                print(f"Missing: {assoc_path.name}")
                continue

            # Add analysis labels before combining results
            df = pd.read_csv(assoc_path, sep='\t')
            df['gene'] = gene
            df['variant_class'] = variant_class
            df['freqUpper'] = maf
            df['ancestry'] = ancestry
            merged_results.append(df)

            print(f"Loaded: {assoc_path.name}")

    if not merged_results:
        print(f"No SKAT-O results found for {ancestry}")
        continue

    # Combine all masks into one ancestry-specific results table
    final_df = pd.concat(merged_results, ignore_index=True)
    out_tsv = ancestry_path / f"{ancestry}_{gene}_RVTESTS_SKATO_merged.tsv"
    final_df.to_csv(out_tsv, sep='\t', index=False)

    print(f"Saved: {out_tsv}")

In [ ]:
## Merge SKAT-O and CMC-Wald results

import pandas as pd

gene = "LRRK2"
mafs = [0.01]
variant_classes = ["potentially_functional", "coding_variants", "loss_of_function", "CADD_gt20"]
ancestries = ["AFR", "AAC", "AJ", "AMR", "CAS", "EAS", "EUR", "MDE", "SAS", "CAH"]
test_types = ["SkatO"]

for ancestry in ancestries:
    ancestry_path = PROJECT_DIR / ancestry
    merged_results = []

    for maf in mafs:
        freq_label = f"{maf:g}"

        for variant_class in variant_classes:
            for test_type in test_types:
                # Locate the RVTests result for this ancestry, mask and test
                assoc_path = ancestry_path / f"{ancestry}.freq{freq_label}.burden.{gene}.{variant_class}.{test_type}.assoc"

                if not assoc_path.exists():
                    print(f"Missing: {assoc_path.name}")
                    continue

                # Add analysis labels before merging
                df = pd.read_csv(assoc_path, sep="\t")
                df["gene"] = gene
                df["variant_class"] = variant_class
                df["freqUpper"] = maf
                df["ancestry"] = ancestry
                df["test_type"] = test_type
                merged_results.append(df)

                print(f"Loaded: {assoc_path.name}")

    if not merged_results:
        print(f"No RVTests results found for {ancestry}")
        continue

    # Combine all masks and tests into one ancestry-specific file
    final_df = pd.concat(merged_results, ignore_index=True)
    out_tsv = ancestry_path / f"{ancestry}_{gene}_RVTESTS_SKATO_merged.tsv"
    final_df.to_csv(out_tsv, sep="\t", index=False)

    print(f"Saved: {out_tsv}")

In [ ]:
## Collect nominally significant SKAT-O results across ancestries

import pandas as pd
from IPython.display import display

gene = 'LRRK2'
ancestries = ['AFR', 'AAC', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'MDE', 'SAS', 'CAH']
pval_cols = ['P', 'Pvalue', 'PVALUE', 'P_VALUE', 'pval', 'p.value']
keepers = []

for ancestry in ancestries:
    # Read the ancestry-level merged SKAT-O results
    path = PROJECT_DIR / ancestry / f"{ancestry}_{gene}_RVTESTS_SKATO_merged.tsv"

    if not path.exists():
        print(f"Missing: {path.name}")
        continue

    df = pd.read_csv(path, sep='\t')
    pcol = next((col for col in pval_cols if col in df.columns), None)

    if pcol is None:
        print(f"No p-value column: {path.name}")
        continue

    # Keep nominally significant results
    df[pcol] = pd.to_numeric(df[pcol], errors='coerce')
    sig = df[df[pcol] < 0.05].copy()

    if sig.empty:
        print(f"No p < 0.05: {path.name}")
        continue

    keepers.append(sig)
    print(f"Loaded {len(sig)} significant row(s): {path.name}")

if keepers:
    # Combine significant results across ancestries
    all_sig = pd.concat(keepers, ignore_index=True)
    out_tsv = PROJECT_DIR / f"ALL_{gene}_SKATO_p_lt_0.05.tsv"
    all_sig.to_csv(out_tsv, sep='\t', index=False)

    print(f"\nSaved: {out_tsv}")
    display(all_sig.head())
    display(all_sig['ancestry'].value_counts().to_frame('N_sig_rows'))
else:
    print("\nNo SKAT-O results with p < 0.05")

In [ ]:
import pandas as pd
from IPython.display import display

gene = 'LRRK2'
ancestries = ['AFR', 'AAC', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'MDE', 'SAS', 'CAH']
pval_cols = ['P', 'Pvalue', 'PVALUE', 'P_VALUE', 'pval', 'p.value']
keepers = []

for ancestry in ancestries:
    # Combined ancestry file contains SKAT-O results
    path = PROJECT_DIR / ancestry / f"{ancestry}_{gene}_RVTESTS_SKATO_merged.tsv"

    if not path.exists():
        print(f"Missing: {path.name}")
        continue

    df = pd.read_csv(path, sep='\t')
    pcol = next((col for col in pval_cols if col in df.columns), None)

    if pcol is None:
        print(f"No p-value column: {path.name}")
        continue

    # Convert p-values to numeric and keep nominally significant results
    df[pcol] = pd.to_numeric(df[pcol], errors='coerce')
    sig = df[df[pcol] < 0.05].copy()

    if sig.empty:
        print(f"No p < 0.05: {path.name}")
        continue

    sig['ancestry'] = ancestry
    sig.rename(columns={pcol: 'Pval'}, inplace=True)
    keepers.append(sig)
    print(f"Loaded {len(sig)} significant row(s): {path.name}")

if keepers:
    # Combine significant SKAT-O results across ancestries
    all_sig = pd.concat(keepers, ignore_index=True)
    out_tsv = PROJECT_DIR / f"ALL_{gene}_SKATO_p_lt_0.05.tsv"
    all_sig.to_csv(out_tsv, sep='\t', index=False)

    print(f"\nSaved: {out_tsv}")
    display(all_sig.head())
    display(all_sig['ancestry'].value_counts().to_frame('N_sig_rows'))
else:
    print("\nNo SKAT-O results with p < 0.05")

In [ ]:
# Merge all SKAT-O results across ancestries

import pandas as pd
from IPython.display import display

gene = "LRRK2"
ancestries = ["AFR", "AAC", "AJ", "AMR", "CAH", "CAS", "EAS", "EUR", "MDE", "SAS"]
skato_tables = []

for ancestry in ancestries:
    path = PROJECT_DIR / ancestry / f"{ancestry}_{gene}_RVTESTS_SKATO_merged.tsv"

    if not path.exists():
        print(f"{ancestry}: file missing")
        continue

    # Read all SKAT-O masks for this ancestry
    skato = pd.read_csv(path, sep="\t")

    if skato.empty:
        print(f"{ancestry}: no SKAT-O rows")
        continue

    # Keep ancestry as the first column
    skato["ancestry"] = ancestry
    skato = skato[["ancestry"] + [column for column in skato.columns if column != "ancestry"]]
    skato_tables.append(skato)

    print(f"{ancestry}: {len(skato)} rows loaded")

if skato_tables:
    # Combine SKAT-O results from all ancestries
    all_skato = pd.concat(skato_tables, ignore_index=True)
    output_path = PROJECT_DIR / f"ALL_{gene}_SKATO_all_ancestries.tsv"
    all_skato.to_csv(output_path, sep="\t", index=False)

    print(f"\nSaved: {output_path}")
    print(f"Total rows: {len(all_skato)}")
    display(all_skato.head())
    display(all_skato.groupby("ancestry").size().to_frame("N_SKATO_rows"))
else:
    print("\nNo SKAT-O results found")

## Part 2. Pathway Burden Analysis: ***LRRK2***, ***RAB32*** and ***VPS35***

This section extends the single-gene workflow to a predefined three-gene set containing ***LRRK2***, ***RAB32*** and ***VPS35***

The genes are extracted separately because they are located on different chromosomes. The extracted genotype files are then merged within each ancestry. Variants are annotated and filtered using the same annotation masks as in the single-gene analysis.

For the final test, all variants from the three genes are grouped into one pathway-level set. Therefore, the pathway analysis produces one SKAT-O result per ancestry and annotation mask, rather than three separate gene-level results.

The analysis uses GRCh38 coordinates:

- *LRRK2*: chromosome 12, 40,224,997–40,369,285
- *RAB32*: chromosome 6, 146,543,833–146,554,953
- *VPS35*: chromosome 16, 46,656,132–46,689,178

In [ ]:
import os
import pathlib

## Define pathway analysis

genes = ["LRRK2", "RAB32", "VPS35"]
pathway_name = "_".join(genes)

ancestries = ["AFR", "AAC", "AJ", "AMR", "CAS", "EAS", "EUR", "MDE", "SAS", "CAH"]
variant_classes = ["potentially_functional", "coding_variants", "loss_of_function", "CADD_gt20"]
freqs = [0.01]

# GRCh38 gene coordinates
gene_regions = {
    "LRRK2": {"chromosome": 12, "start": 40_224_997,  "end": 40_369_285},
    "RAB32": {"chromosome": 6,  "start": 146_543_833, "end": 146_554_953},
    "VPS35": {"chromosome": 16, "start": 46_656_132,  "end": 46_689_178},}

# Create the main trainee directory (if it doesn't already exist)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Create a pathway-specific directory
PATHWAY_DIR = WORK_DIR / pathway_name
PATHWAY_DIR.mkdir(parents=True, exist_ok=True)

# Create one folder per ancestry
for ancestry in ancestries:
    (PATHWAY_DIR / ancestry).mkdir(parents=True, exist_ok=True)

print(f"Trainee ID: {trainee_id}")
print(f"Pathway: {pathway_name}")
print(f"Pathway directory: {PATHWAY_DIR}")

In [ ]:
# Move covariate from previous analysis to folder above

import shutil
from pathlib import Path

source_dir = Path(
    f"/home/jupyter/workspace/ws_files/BOTH_LEVELS_WORKSHOP/"
    f"BURDEN_ANALYSIS_{trainee_id}/LRRK2")

target_dir = Path(
    f"/home/jupyter/workspace/ws_files/BOTH_LEVELS_WORKSHOP/"
    f"BURDEN_ANALYSIS_{trainee_id}/LRRK2_RAB32_VPS35")

for ancestry in ancestries:
    source_cov = source_dir / ancestry / f"{ancestry}_covariate.txt"
    target_cov = target_dir / ancestry / f"{ancestry}_covariate.txt"

    target_cov.parent.mkdir(parents=True, exist_ok=True)

    if not source_cov.exists():
        print(f"Missing: {source_cov}")
        continue

    shutil.copy2(source_cov, target_cov)
    print(f"Copied: {ancestry}_covariate.txt")


In [ ]:
## Extract each pathway gene using PLINK

for ancestry in ancestries:
    ancestry_path = PATHWAY_DIR / ancestry

    for gene_name, region in gene_regions.items():
        chromosome = region['chromosome']
        gene_start = region['start']
        gene_end = region['end']

        input_prefix = WGS_DATA_PATH / ancestry / f"chr{chromosome}_{ancestry}_release12"
        output_prefix = ancestry_path / f"{ancestry}_{gene_name}"

        print(f"\nExtracting {gene_name} in {ancestry}")

        # Check that the chromosome-level GP2 files are available
        if not input_prefix.with_suffix('.pgen').exists():
            print(f"Missing: {input_prefix}.pgen")
            continue

        # Extract the GRCh38 gene region into a separate PLINK fileset
        ! /home/jupyter/plink2 \
        --pfile {input_prefix} \
        --chr {chromosome} \
        --from-bp {gene_start} \
        --to-bp {gene_end} \
        --make-pgen \
        --out {output_prefix}

In [ ]:
## Check extracted pathway files

for ancestry in ancestries:
    ancestry_path = PATHWAY_DIR / ancestry

    for gene_name in genes:
        prefix = ancestry_path / f"{ancestry}_{gene_name}"
        required_files = [prefix.with_suffix('.pgen'), prefix.with_suffix('.pvar'), prefix.with_suffix('.psam')]
        missing = [file.name for file in required_files if not file.exists()]

        if missing:
            print(f"{ancestry}, {gene_name}: missing {', '.join(missing)}")
        else:
            print(f"{ancestry}, {gene_name}: files ready")

In [ ]:
## Merge pathway genes within each ancestry

for ancestry in ancestries:
    ancestry_path = PATHWAY_DIR / ancestry
    base_prefix = ancestry_path / f"{ancestry}_{genes[0]}"
    merge_list = ancestry_path / f"{pathway_name}_pmerge_list.txt"
    merged_prefix = ancestry_path / f"{ancestry}_{pathway_name}_merged"

    print(f"\nMerging pathway genes in {ancestry}")

    # Confirm that all three gene files are available
    missing_genes = [
        gene_name for gene_name in genes
        if not (ancestry_path / f"{ancestry}_{gene_name}").with_suffix('.pgen').exists()
    ]

    if missing_genes:
        print(f"Missing gene files: {', '.join(missing_genes)}")
        continue

    # The first gene is supplied with --pfile; the remaining genes go into the merge list
    with open(merge_list, 'w') as file:
        for gene_name in genes[1:]:
            file.write(f"{ancestry_path / f'{ancestry}_{gene_name}'}\n")

    # Merge non-overlapping regions while retaining samples present in every fileset
    ! /home/jupyter/plink2 \
    --pfile {base_prefix} \
    --pmerge-list {merge_list} \
    --sample-inner-join \
    --make-pgen \
    --out {merged_prefix}

    if merged_prefix.with_suffix('.pgen').exists():
        print(f"{ancestry}: pathway merge completed")
    else:
        print(f"{ancestry}: pathway merge failed")

In [ ]:
## Export merged pathway files to compressed VCF

for ancestry in ancestries:
    ancestry_path = PATHWAY_DIR / ancestry
    input_prefix = ancestry_path / f"{ancestry}_{pathway_name}_merged"
    output_prefix = ancestry_path / f"{ancestry}_{pathway_name}"

    print(f"\nExporting pathway VCF for {ancestry}")

    if not input_prefix.with_suffix('.pgen').exists():
        print(f"Missing: {input_prefix}.pgen")
        continue

    # Keep the same sample-ID and dosage format used in the single-gene analysis
    ! /home/jupyter/plink2 \
    --pfile {input_prefix} \
    --export vcf id-paste=iid vcf-dosage=HDS bgz \
    --out {output_prefix}

In [ ]:
## Index pathway VCF files

for ancestry in ancestries:
    vcf_path = PATHWAY_DIR / ancestry / f"{ancestry}_{pathway_name}.vcf.gz"

    if not vcf_path.exists():
        print(f"Missing: {vcf_path}")
        continue

    # RVTests requires indexed VCF files for group-based analyses
    if os.system(f"tabix -f -p vcf {vcf_path}") != 0:
        print(f"Indexing failed: {ancestry}")
    else:
        print(f"Indexed: {vcf_path.name}")

In [ ]:
## Annotate pathway variants with ANNOVAR

# Run programs locally but use the shared reference databases
ANNOVAR_DIR = pathlib.Path("/home/jupyter/annovar")
ANNOVAR_DB = pathlib.Path(
    "/home/jupyter/workspace/ws_files/Reference/annovar/humandb"
)

for ancestry in ancestries:

    ancestry_path = PATHWAY_DIR / ancestry
    vcf_path = ancestry_path / f"{ancestry}_{pathway_name}.vcf.gz"
    avinput_path = ancestry_path / f"{ancestry}_{pathway_name}.annovar.avinput"
    out_prefix = ancestry_path / f"{ancestry}_{pathway_name}.annovar"

    print(f"\nAnnotating {ancestry}")

    if not vcf_path.exists():
        print(f"Missing VCF: {vcf_path}")
        continue

    # Decompress the VCF and convert it to ANNOVAR input format
    convert_cmd = (
        f'gzip -dc "{vcf_path}" | '
        f'perl "{ANNOVAR_DIR}/convert2annovar.pl" '
        f'-format vcf4old - > "{avinput_path}"'
    )

    # Add gene consequences and dbNSFP prediction scores
    annotate_cmd = (
        f'perl "{ANNOVAR_DIR}/table_annovar.pl" '
        f'"{avinput_path}" "{ANNOVAR_DB}" '
        f'-buildver hg38 '
        f'-out "{out_prefix}" '
        f'-remove '
        f'-protocol refGene,dbnsfp47a '
        f'-operation g,f '
        f'-nastring . '
        f'-polish'
    )

    if os.system(convert_cmd) != 0:
        print(f"Conversion failed for {ancestry}")
        continue

    if os.system(annotate_cmd) != 0:
        print(f"Annotation failed for {ancestry}")
        continue

    print(f"Annotation completed for {ancestry}")

In [ ]:
## Check VCF and ANNOVAR variant counts

for ancestry in ancestries:
    ancestry_path = PATHWAY_DIR / ancestry
    vcf_path = ancestry_path / f"{ancestry}_{pathway_name}.vcf.gz"
    avinput_path = ancestry_path / f"{ancestry}_{pathway_name}.annovar.avinput"

    vcf_count = int(subprocess.check_output(
        f"bcftools view -H {vcf_path} | wc -l",
        shell=True,
        text=True
    ).strip())

    avinput_count = sum(1 for _ in open(avinput_path))

    print(f"{ancestry}: VCF={vcf_count}, ANNOVAR={avinput_count}")

In [ ]:
## Create annotation-based pathway masks

gene_pattern = rf"(?:^|[;,])(?:{'|'.join(genes)})(?:$|[;,])"
cols = ['Chr', 'Start', 'End', 'Gene.refGene']

for ancestry in ancestries:
    ancestry_path = PATHWAY_DIR / ancestry
    file_path = ancestry_path / f"{ancestry}_{pathway_name}.annovar.hg38_multianno.txt"
    print(f"\nWorking on {ancestry}, {pathway_name}")

    if not file_path.exists():
        print(f"Missing file: {file_path}")
        continue

    # Retain annotations assigned to one of the three pathway genes
    pathway_df = pd.read_csv(file_path, sep='\t')
    pathway_df = pathway_df[pathway_df['Gene.refGene'].str.contains(gene_pattern, regex=True, na=False)].copy()
    pathway_df['CADD_phred'] = pd.to_numeric(pathway_df['CADD_phred'], errors='coerce')
    print(f"Total variants: {len(pathway_df)}")

    # Apply the same four masks used in the single-gene analysis
    potentially_functional = pathway_df[pathway_df['Func.refGene'] != 'intronic']
    coding_variants = pathway_df[(pathway_df['Func.refGene'] == 'splicing') | ((pathway_df['Func.refGene'] == 'exonic') & (pathway_df['ExonicFunc.refGene'] != 'synonymous SNV'))]
    loss_of_function = pathway_df[(pathway_df['Func.refGene'] == 'splicing') | pathway_df['ExonicFunc.refGene'].isin(['stopgain', 'startloss', 'frameshift deletion', 'frameshift insertion'])]
    cadd_gt20 = pathway_df[pathway_df['CADD_phred'] > 20]

    prefix = ancestry_path / f"{ancestry}_{pathway_name}"
    masks = {'potentially_functional': potentially_functional, 'coding_variants': coding_variants, 'loss_of_function': loss_of_function, 'CADD_gt20': cadd_gt20}

    # Remove repeated genomic positions before writing PLINK range files
    for mask_name, mask_df in masks.items():
        variants = mask_df[cols].drop_duplicates(subset=['Chr', 'Start', 'End'])
        variants.to_csv(f"{prefix}.{mask_name}.variantstoKeep.txt", sep='\t', index=False, header=False)
        print(f"{mask_name}: {len(variants)} variants")

In [ ]:
## Extract pathway VCFs for each annotation mask

for ancestry in ancestries:
    ancestry_path = PATHWAY_DIR / ancestry
    vcf_input = ancestry_path / f"{ancestry}_{pathway_name}.vcf.gz"

    if not vcf_input.exists():
        print(f"Missing VCF: {vcf_input}")
        continue

    for variant_class in variant_classes:
        extract_file = ancestry_path / f"{ancestry}_{pathway_name}.{variant_class}.variantstoKeep.txt"
        output_prefix = ancestry_path / f"{ancestry}_{pathway_name}.{variant_class}"

        print(f"\nExtracting {ancestry}, {variant_class}")

        if not extract_file.exists() or extract_file.stat().st_size == 0:
            print(f"No variants: {extract_file}")
            continue

        # Extract all variants in the mask across the three pathway genes
        cmd = (
            f"/home/jupyter/plink2 "
            f"--vcf {vcf_input} dosage=HDS "
            f"--extract range {extract_file} "
            f"--export vcf id-paste=iid vcf-dosage=HDS bgz "
            f"--out {output_prefix}"
        )

        if os.system(cmd) != 0:
            print(f"Extraction failed: {ancestry}, {variant_class}")
            continue

        if os.system(f"tabix -f -p vcf {output_prefix}.vcf.gz") != 0:
            print(f"Indexing failed: {ancestry}, {variant_class}")

## Define a pathway-level grouping unit

* RVTests normally groups variants by individual genes using `--geneFile`. For this analysis, the three genes must be treated as one combined burden unit.
* A set file defines one named set containing the genomic ranges of *LRRK2*, *RAB32* and *VPS35*. RVTests will test all eligible variants across these ranges together.

In [ ]:
## Create an RVTests pathway set file

set_ranges = ','.join(
    f"{region['chromosome']}:{region['start']}-{region['end']}"
    for region in gene_regions.values()
)

pathway_set_file = PATHWAY_DIR / f"{pathway_name}.set"
pathway_set_file.write_text(f"{pathway_name}\t{set_ranges}\n")

print(pathway_set_file.read_text())

In [ ]:
## Run pathway-level SKAT-O

import os

genes = ['LRRK2', 'RAB32', 'VPS35']
pathway_name = '_'.join(genes)
freqs = [0.01]
variant_classes = ['potentially_functional', 'coding_variants', 'loss_of_function', 'CADD_gt20']
ancestries = ['AFR', 'AAC', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'MDE', 'SAS', 'CAH']

# Covariate files were prepared during the LRRK2 single-gene analysis
COVARIATE_DIR = pathlib.Path(WORK_DIR) / 'LRRK2'
pathway_set_file = PATHWAY_DIR / f"{pathway_name}.set"

if not pathway_set_file.exists():
    raise FileNotFoundError(f"Missing pathway set file: {pathway_set_file}")

for ancestry in ancestries:
    ancestry_path = PATHWAY_DIR / ancestry
    pheno_file = COVARIATE_DIR / ancestry / f"{ancestry}_covariate.txt"

    for freq in freqs:
        freq_label = f"{freq:g}"

        for variant_class in variant_classes:
            vcf_input = ancestry_path / f"{ancestry}_{pathway_name}.{variant_class}.vcf.gz"
            output_prefix = ancestry_path / f"{ancestry}.freq{freq_label}.pathway.{pathway_name}.{variant_class}"

            print(f"\nRunning {ancestry}, {variant_class}, MAF ≤ {freq}")

            if not vcf_input.exists():
                print(f"Missing VCF: {vcf_input}")
                continue

            if not pheno_file.exists():
                print(f"Missing covariate file: {pheno_file}")
                continue

            # Test all three genes together as one pathway-level variant set
            cmd = (
                f"/home/jupyter/tools/rvtests/executable/rvtest "
                f"--noweb --hide-covar "
                f"--out {output_prefix} "
                f"--kernel skato "
                f"--freqUpper {freq} "
                f"--inVcf {vcf_input} "
                f"--pheno {pheno_file} "
                f"--pheno-name PHENO "
                f"--setFile {pathway_set_file} "
                f"--set {pathway_name} "
                f"--covar {pheno_file} "
                f"--covar-name SEX,AGE,PC1,PC2,PC3,PC4,PC5"
            )

            if os.system(cmd) != 0:
                print(f"RVTests failed: {ancestry}, {variant_class}")

In [ ]:
## Merge pathway SKAT-O results across annotation masks

import pandas as pd

genes = ['LRRK2', 'RAB32', 'VPS35']
pathway_name = '_'.join(genes)
mafs = [0.01]
variant_classes = ['potentially_functional', 'coding_variants', 'loss_of_function', 'CADD_gt20']
ancestries = ['AFR', 'AAC', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'MDE', 'SAS', 'CAH']

for ancestry in ancestries:
    ancestry_path = PATHWAY_DIR / ancestry
    merged_results = []

    for maf in mafs:
        freq_label = f"{maf:g}"

        for variant_class in variant_classes:
            assoc_path = ancestry_path / f"{ancestry}.freq{freq_label}.pathway.{pathway_name}.{variant_class}.SkatO.assoc"

            if not assoc_path.exists():
                print(f"Missing: {assoc_path.name}")
                continue

            # Add pathway-analysis labels before combining masks
            df = pd.read_csv(assoc_path, sep='\t')
            df['pathway'] = pathway_name
            df['genes'] = ','.join(genes)
            df['variant_class'] = variant_class
            df['freqUpper'] = maf
            df['ancestry'] = ancestry
            merged_results.append(df)

            print(f"Loaded: {assoc_path.name}")

    if not merged_results:
        print(f"No pathway SKAT-O results found for {ancestry}")
        continue

    final_df = pd.concat(merged_results, ignore_index=True)
    out_tsv = ancestry_path / f"{ancestry}_{pathway_name}_PATHWAY_SKATO_merged.tsv"
    final_df.to_csv(out_tsv, sep='\t', index=False)

    print(f"Saved: {out_tsv}")

In [ ]:
## Merge pathway SKAT-O results across ancestries

import pandas as pd
from IPython.display import display

genes = ['LRRK2', 'RAB32', 'VPS35']
pathway_name = '_'.join(genes)
ancestries = ['AFR', 'AAC', 'AJ', 'AMR', 'CAH', 'CAS', 'EAS', 'EUR', 'MDE', 'SAS']
pathway_tables = []

for ancestry in ancestries:
    path = PATHWAY_DIR / ancestry / f"{ancestry}_{pathway_name}_PATHWAY_SKATO_merged.tsv"

    if not path.exists():
        print(f"{ancestry}: file missing")
        continue

    pathway_df = pd.read_csv(path, sep='\t')

    if pathway_df.empty:
        print(f"{ancestry}: no pathway SKAT-O rows")
        continue

    pathway_df['ancestry'] = ancestry
    pathway_df = pathway_df[['ancestry'] + [column for column in pathway_df.columns if column != 'ancestry']]
    pathway_tables.append(pathway_df)

    print(f"{ancestry}: {len(pathway_df)} rows loaded")

if pathway_tables:
    all_pathway = pd.concat(pathway_tables, ignore_index=True)
    output_path = PATHWAY_DIR / f"ALL_{pathway_name}_PATHWAY_SKATO.tsv"
    all_pathway.to_csv(output_path, sep='\t', index=False)

    print(f"\nSaved: {output_path}")
    print(f"Total rows: {len(all_pathway)}")
    display(all_pathway.head())
    display(all_pathway.groupby('ancestry').size().to_frame('N_SKATO_rows'))
else:
    print("\nNo pathway SKAT-O results found")